In [1]:
# simple toy example to observe state message orchestration

In [14]:
import random
from typing import TypedDict, Literal
from enum import Enum

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

In [ ]:
class State(TypedDict):
    random_number: int
    message: str
    messages: list[str]


class NodeName(str, Enum):
    GENERATE_NUMBER = "generate_number"
    PRINT_MULTIPLE_3 = "print_multiple_3"
    PRINT_NOT_MULTIPLE_3 = "print_not_multiple_3"

In [ ]:
def generate_number(state: State) -> Command[Literal[NodeName.PRINT_MULTIPLE_3, NodeName.PRINT_NOT_MULTIPLE_3]]:
    """Node 1: Generate a random number and route conditionally"""
    
    print("\n----> Now in node1")
    
    num = random.randint(1, 100)
    print(f"Generated random number: {num}")
    
    # Create message for this node
    node_message = f"Node 1: Generated number {num}"
    
    # Append to existing messages
    messages = state.get("messages", [])
    messages.append(node_message)
    
    # Route based on whether it's a multiple of 3
    if num % 3 == 0:
        next_node = NodeName.PRINT_MULTIPLE_3
    else:
        next_node = NodeName.PRINT_NOT_MULTIPLE_3
    
    
    return Command(
        update={
            "random_number": num, 
            "message": f"Generated number: {num}",
            "messages": messages
        },
        goto=next_node
    )


def print_multiple_3(state: State) -> Command[Literal[END]]:
    """Node 2: Print message for multiples of 3"""
    
    print("\n----> Now in node2")
    
    print(f" It is a multiple of 3.")
    # print(f"Message: {state['message']}")
    
    # Create message for this node
    node_message = f"Node 2: {state['random_number']} is a multiple of 3!"
    
    # Append to existing messages
    messages = state.get("messages", [])
    messages.append(node_message)
    
    
    return Command(
        update={
            "message": f"{state['random_number']} is a multiple of 3!",
            "messages": messages
        },
        goto=END
    )


def print_not_multiple_3(state: State) -> Command[Literal[END]]:
    """Node 3: Print message for non-multiples of 3"""
    
    print("\n----> Now in node3")
    
    print(f"It is NOT a multiple of 3.")
    print(f"Message: {state['message']}")
    
    # Create message for this node
    node_message = f"Node 3: {state['random_number']} is NOT a multiple of 3!"
    
    # Append to existing messages
    messages = state.get("messages", [])
    messages.append(node_message)
        
    return Command(
        update={
            "message": f"{state['random_number']} is NOT a multiple of 3!",
            "messages": messages
        },
        goto=END
    )

In [29]:
def create_graph() -> StateGraph:
    """Create the conditional routing graph"""
    graph = StateGraph(State)
    
    # Add nodes
    graph.add_node(NodeName.GENERATE_NUMBER, generate_number)
    graph.add_node(NodeName.PRINT_MULTIPLE_3, print_multiple_3)
    graph.add_node(NodeName.PRINT_NOT_MULTIPLE_3, print_not_multiple_3)
    
    # Add edges
    graph.add_edge(START, NodeName.GENERATE_NUMBER)
    
    return graph.compile()


In [56]:
def run_simple_graph():
    """Run the simple graph example"""
    graph = create_graph()
    
    # Initial state
    initial_state = {"random_number": 0, "message": "Starting...", "messages": []}
    
    print(" Running Simple Conditional Graph")
    print("=" * 50)
    
    # Run the graph
    result = graph.invoke(initial_state)
    
    print("\n\n Final State:")
    print(f"a) Random Number: {result['random_number']}")
    print(f"b) Message: {result['message']}")
    
    print("c) All Messages from Graph Execution:")
    print("\t", "-" * 40)
    for i, msg in enumerate(result['messages'], 1):
        print(f"\t{i}. {msg}")
    
    return result


In [57]:
if __name__ == "__main__":
    # Run multiple times to see different routing
    
    num_cnts = 1
    for i in range(num_cnts):
        print(f"\n Run {i+1}:")
        res = run_simple_graph()



 Run 1:
 Running Simple Conditional Graph

----> Now in node1
Generated random number: 38

----> Now in node3
It is NOT a multiple of 3.
Message: Generated number: 38


 Final State:
a) Random Number: 38
b) Message: 38 is NOT a multiple of 3!
c) All Messages from Graph Execution:
	 ----------------------------------------
	1. Node 1: Generated number 38
	2. Node 3: 38 is NOT a multiple of 3!


In [58]:
res

{'random_number': 38,
 'message': '38 is NOT a multiple of 3!',
 'messages': ['Node 1: Generated number 38',
  'Node 3: 38 is NOT a multiple of 3!']}